# NLP Concepts Walkthrough
### From raw text to structured intelligence — step by step

This notebook walks through the core NLP concepts powering the two demos
in this portfolio. Each section explains the concept, shows the code, and
connects it to real-world use cases.

**Concepts covered:**
1. Text preprocessing
2. Named Entity Recognition (NER)
3. Keyword extraction (TF-IDF)
4. Sentiment analysis
5. The full pipeline — putting it all together
6. What's next — upgrading to production

In [1]:
# Core NLP
import spacy
from collections import Counter
import math
import re

# Load the English model
nlp = spacy.load("en_core_web_sm")

# Sample texts used throughout the notebook
news_text = """
Apple Inc. reported record quarterly revenue of $123.9 billion in Q1 2024,
driven by strong iPhone 15 sales in China and India. CEO Tim Cook praised
the company's performance in emerging markets but warned that supply chain
disruptions in Taiwan could impact next quarter's output.
"""

legal_text = """
This Agreement shall be governed by the laws of the State of New York.
The parties agree to submit to binding arbitration in New York County.
The Company shall indemnify and hold harmless Employee from any claims
arising from acts performed in good faith within the scope of employment.
"""

print("✅ Libraries loaded successfully")
print(f"spaCy version: {spacy.__version__}")

✅ Libraries loaded successfully
spaCy version: 3.8.13


## 1. Text Preprocessing

Before any NLP task, raw text needs to be cleaned and tokenized.
Tokenization splits text into individual units (tokens) — words, punctuation, numbers.
spaCy does this automatically and also handles:
- **Lemmatization** — reducing words to their base form (running → run)
- **Stop word removal** — filtering out common words (the, is, at) that carry little meaning
- **Part-of-speech tagging** — labeling each token as noun, verb, adjective, etc.

In [2]:
doc = nlp(news_text)

print("=== TOKENS (first 15) ===")
print(f"{'Token':<20} {'Lemma':<20} {'POS':<10} {'Stop?':<8}")
print("-" * 60)
for token in list(doc)[:15]:
    if not token.is_space:
        print(f"{token.text:<20} {token.lemma_:<20} {token.pos_:<10} {str(token.is_stop):<8}")

print(f"\nTotal tokens: {len(doc)}")
print(f"Sentences found: {len(list(doc.sents))}")

=== TOKENS (first 15) ===
Token                Lemma                POS        Stop?   
------------------------------------------------------------
Apple                Apple                PROPN      False   
Inc.                 Inc.                 PROPN      False   
reported             report               VERB       False   
record               record               ADJ        False   
quarterly            quarterly            ADJ        False   
revenue              revenue              NOUN       False   
of                   of                   ADP        True    
$                    $                    SYM        False   
123.9                123.9                NUM        False   
billion              billion              NUM        False   
in                   in                   ADP        True    
Q1                   q1                   NOUN       False   
2024                 2024                 NUM        False   
,                    ,                    PUN

## 2. Named Entity Recognition (NER)

NER identifies and classifies real-world entities in text —
people, organizations, locations, dates, monetary values, and more.

This is one of the most valuable NLP tasks for business applications —
it's how you extract structured data from unstructured documents.

**Common entity types:**
- `PERSON` — people, fictional characters
- `ORG` — companies, agencies, institutions
- `GPE` — countries, cities, states
- `MONEY` — monetary values
- `DATE` — dates and time periods

In [3]:
doc = nlp(news_text)

print("=== NAMED ENTITIES ===")
print(f"{'Entity':<30} {'Type':<12} {'Description'}")
print("-" * 70)

type_desc = {
    "PERSON": "Person name",
    "ORG": "Organization",
    "GPE": "Country/city/state",
    "LOC": "Location",
    "MONEY": "Monetary value",
    "DATE": "Date or period",
    "PERCENT": "Percentage",
    "PRODUCT": "Product name",
}

seen = set()
for ent in doc.ents:
    if ent.text.strip() not in seen:
        desc = type_desc.get(ent.label_, ent.label_)
        print(f"{ent.text.strip():<30} {ent.label_:<12} {desc}")
        seen.add(ent.text.strip())

print(f"\nTotal unique entities: {len(seen)}")

=== NAMED ENTITIES ===
Entity                         Type         Description
----------------------------------------------------------------------
Apple Inc.                     ORG          Organization
quarterly                      DATE         Date or period
$123.9 billion                 MONEY        Monetary value
Q1 2024                        DATE         Date or period
15                             CARDINAL     CARDINAL
China                          GPE          Country/city/state
India                          GPE          Country/city/state
Tim Cook                       PERSON       Person name
Taiwan                         GPE          Country/city/state
next quarter's                 DATE         Date or period

Total unique entities: 10


## 3. Keyword Extraction with TF-IDF

**TF-IDF** (Term Frequency–Inverse Document Frequency) ranks how important
a word is to a document relative to a collection of documents.

- **TF** — how often a word appears in *this* document
- **IDF** — how rare the word is *across all documents*

A word appearing often in one doc but rarely across others = high importance.
Common words like "the" appear everywhere = low importance.

In [4]:
def compute_tfidf(documents):
    N = len(documents)

    def tokenize(text):
        doc = nlp(text.lower())
        return [t.lemma_ for t in doc
                if not t.is_stop and not t.is_punct
                and not t.is_space and len(t.text) > 2]

    tokenized = [tokenize(d) for d in documents]

    df = Counter()
    for tokens in tokenized:
        for word in set(tokens):
            df[word] += 1

    tf = Counter(tokenized[0])
    total_tokens = len(tokenized[0])

    tfidf = {}
    for word, count in tf.items():
        term_freq = count / total_tokens
        inv_doc_freq = math.log(N / (df[word] + 1))
        tfidf[word] = round(term_freq * inv_doc_freq, 4)

    return sorted(tfidf.items(), key=lambda x: x[1], reverse=True)

corpus = [news_text, legal_text]
scores = compute_tfidf(corpus)

print("=== TOP KEYWORDS BY TF-IDF (news article) ===")
print(f"{'Keyword':<20} {'Score':<10} {'Bar'}")
print("-" * 50)
for word, score in scores[:10]:
    bar = "█" * int(score * 500)
    print(f"{word:<20} {score:<10} {bar}")

=== TOP KEYWORDS BY TF-IDF (news article) ===
Keyword              Score      Bar
--------------------------------------------------
apple                0.0        
inc                  0.0        
report               0.0        
record               0.0        
quarterly            0.0        
revenue              0.0        
123.9                0.0        
billion              0.0        
2024                 0.0        
drive                0.0        


## 4. Sentiment Analysis

Sentiment analysis classifies the emotional tone of text.

**Two main approaches:**
- **Rule-based** — lexicons of positive/negative words (fast, interpretable)
- **Transformer-based** — BERT/RoBERTa fine-tuned on labeled data (accurate)

Below is a simple rule-based implementation to show the concept clearly.

In [5]:
POSITIVE_WORDS = {
    "record", "strong", "praised", "growth", "success", "best",
    "excellent", "outstanding", "exceptional", "robust", "surpassed"
}
NEGATIVE_WORDS = {
    "disruption", "disruptions", "warned", "risk", "risks", "decline",
    "fell", "loss", "concern", "concerns", "failed", "problem"
}

def simple_sentiment(text):
    doc = nlp(text.lower())
    tokens = [t.lemma_ for t in doc if not t.is_stop and not t.is_punct]

    pos = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    total = pos + neg

    if total == 0:
        return {"label": "neutral", "score": 0.5, "pos": 0, "neg": 0}

    score = pos / total
    if score > 0.6:
        label = "positive"
    elif score < 0.4:
        label = "negative"
    elif pos > 0 and neg > 0:
        label = "mixed"
    else:
        label = "neutral"

    return {"label": label, "score": round(score, 2), "pos": pos, "neg": neg}

result = simple_sentiment(news_text)
print("=== SENTIMENT ANALYSIS (news article) ===")
print(f"  Label:         {result['label'].upper()}")
print(f"  Score:         {result['score']}  (1.0 = fully positive)")
print(f"  Positive hits: {result['pos']}")
print(f"  Negative hits: {result['neg']}")
print()
print("⚠️  Rule-based misses context and sarcasm.")
print("✅  Production upgrade:")
print("    from transformers import pipeline")
print("    sentiment = pipeline('sentiment-analysis')")
print("    sentiment(text)  # → {'label': 'POSITIVE', 'score': 0.94}")

=== SENTIMENT ANALYSIS (news article) ===
  Label:         POSITIVE
  Score:         0.67  (1.0 = fully positive)
  Positive hits: 2
  Negative hits: 1

⚠️  Rule-based misses context and sarcasm.
✅  Production upgrade:
    from transformers import pipeline
    sentiment = pipeline('sentiment-analysis')
    sentiment(text)  # → {'label': 'POSITIVE', 'score': 0.94}


## 5. The Full Pipeline — Putting It All Together

This combines every technique above into one function —
mirroring what the FastAPI backend does in Demo 1.

In [6]:
def full_nlp_pipeline(text):
    doc = nlp(text)

    # Step 1: Basic stats
    tokens    = [t for t in doc if not t.is_space]
    sentences = list(doc.sents)
    words     = [t for t in tokens if not t.is_punct]

    # Step 2: Named entities
    entities = [
        {"text": ent.text.strip(), "type": ent.label_}
        for ent in doc.ents if len(ent.text.strip()) > 1
    ]

    # Step 3: Keywords by frequency
    content_words = [
        t.lemma_.lower() for t in doc
        if not t.is_stop and not t.is_punct
        and not t.is_space and len(t.text) > 2
    ]
    top_keywords = Counter(content_words).most_common(8)

    # Step 4: Sentiment
    sentiment = simple_sentiment(text)

    # Step 5: Readability
    avg_sent_len = len(words) / max(len(sentences), 1)
    avg_word_len = sum(len(w.text) for w in words) / max(len(words), 1)
    readability  = round(206.835 - 1.015 * avg_sent_len - 84.6 * avg_word_len, 1)
    read_label   = "Easy" if readability > 70 else "Standard" if readability > 50 else "Complex"

    return {
        "stats": {
            "words":          len(words),
            "sentences":      len(sentences),
            "avg_word_len":   round(avg_word_len, 1),
            "readability":    f"{readability} ({read_label})"
        },
        "sentiment": sentiment,
        "entities":  entities[:8],
        "keywords":  [{"word": w, "count": c} for w, c in top_keywords]
    }

result = full_nlp_pipeline(news_text)

print("=" * 55)
print("FULL NLP PIPELINE OUTPUT — news article")
print("=" * 55)

print("\n📊 TEXT STATISTICS")
for k, v in result["stats"].items():
    print(f"  {k:<22} {v}")

print(f"\n🎭 SENTIMENT: {result['sentiment']['label'].upper()} (score: {result['sentiment']['score']})")

print("\n🏷️  NAMED ENTITIES")
for ent in result["entities"]:
    print(f"  [{ent['type']:>8}]  {ent['text']}")

print("\n🔑 TOP KEYWORDS")
for kw in result["keywords"]:
    print(f"  {kw['word']:<20} count: {kw['count']}")

FULL NLP PIPELINE OUTPUT — news article

📊 TEXT STATISTICS
  words                  48
  sentences              2
  avg_word_len           4.9
  readability            -229.9 (Complex)

🎭 SENTIMENT: POSITIVE (score: 0.67)

🏷️  NAMED ENTITIES
  [     ORG]  Apple Inc.
  [    DATE]  quarterly
  [   MONEY]  $123.9 billion
  [    DATE]  Q1 2024
  [CARDINAL]  15
  [     GPE]  China
  [     GPE]  India
  [  PERSON]  Tim Cook

🔑 TOP KEYWORDS
  apple                count: 1
  inc.                 count: 1
  report               count: 1
  record               count: 1
  quarterly            count: 1
  revenue              count: 1
  123.9                count: 1
  billion              count: 1


## 6. What's Next — Upgrading to Production

| Component | This notebook | Production upgrade |
|-----------|--------------|-------------------|
| Sentiment | Rule-based word lists | Fine-tuned BERT (`cardiffnlp/twitter-roberta-base-sentiment`) |
| NER | spaCy `en_core_web_sm` | spaCy `en_core_web_trf` (transformer-based) |
| Keywords | Word frequency | TF-IDF across corpus, or KeyBERT |
| Classification | Prompt-based (Claude) | Fine-tuned classifier on labeled data |
| Summarization | Prompt-based (Claude) | BART or T5 fine-tuned on domain data |

## Key libraries to learn next

```python
# Hugging Face Transformers
from transformers import pipeline
sentiment  = pipeline("sentiment-analysis")
summarizer = pipeline("summarization")
ner        = pipeline("ner", grouped_entities=True)

# Sentence embeddings
from sentence_transformers import SentenceTransformer
model      = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(["text one", "text two"])

# PyTorch — for fine-tuning custom models
import torch
import torch.nn as nn
```

## Resources
- [spaCy course](https://course.spacy.io) — free, interactive, excellent
- [HuggingFace NLP course](https://huggingface.co/learn/nlp-course) — transformers from scratch
- [CUAD dataset](https://huggingface.co/datasets/cuad) — 500 labeled legal contracts, perfect for Demo 2 fine-tuning